# trainer-subclass-extend — worked example 2: Subclass trainer to apply gradient clipping before optimizer step

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-subclass-extend`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Gradient clipping is often added as a modification to the training step rather than a change to the model architecture. By overriding `_step()` in a subclass and calling `super()._step()`, you can insert a `clip_grad_norm_` call between `backward()` and `step()`. However, `_step` only computes the forward pass and loss — the backward happens in `fit`. A cleaner hook is to override `fit`'s inner loop body, which you can do by overriding the entire `fit` method and inserting the clip. This worked example shows that pattern.

## Worked solution

**Step 1 – Add `max_norm` to `__init__`.** We call `super().__init__(...)` then store `self.max_norm = max_norm` and `self.grad_norms = []` to track what clipping did.

**Step 2 – Override `fit` to add clipping.** We copy the base `fit` structure but insert `nn.utils.clip_grad_norm_(self.model.parameters(), self.max_norm)` AFTER `loss.backward()` and BEFORE `optimizer.step()`. We also record the norm before clipping.

**Step 3 – Record pre-clip norm.** Before the clip call, `total_norm = t.nn.utils.clip_grad_norm_(...)` returns the actual norm. We save it to `self.grad_norms`.

**Step 4 – Call `validate()` as before.** The validate method is inherited unchanged from the base. Only the training step is modified.

In [ ]:
import torch as t
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class BaseTrainer2:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)


class GradClipTrainer(BaseTrainer2):
    """Extends BaseTrainer2 with gradient norm clipping."""
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn, max_norm=1.0):
        super().__init__(model, optimizer, train_loader, val_loader, loss_fn)
        self.max_norm = max_norm
        self.grad_norms = []  # pre-clip norm per step

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                # Insert gradient clipping BEFORE optimizer step
                pre_clip_norm = nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.max_norm
                ).item()
                self.grad_norms.append(pre_clip_norm)
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

# Demo
t.manual_seed(8)
X = t.randn(30, 3)
Y = X[:, 0:1]
train_dl = DataLoader(TensorDataset(X[:24], Y[:24]), batch_size=6)
val_dl = DataLoader(TensorDataset(X[24:], Y[24:]), batch_size=6)
model = nn.Linear(3, 1)
trainer = GradClipTrainer(model, t.optim.SGD(model.parameters(), lr=0.1),
                          train_dl, val_dl, nn.MSELoss(), max_norm=0.5)
trainer.fit(2)
print('grad norms (pre-clip):', [f'{v:.4f}' for v in trainer.grad_norms])
print('max clipped norm:', max(trainer.grad_norms))